# Export model

### Resources

- https://geemap.org/notebooks/46_local_rf_training/

## Setup

In [ ]:
import time

import ee
from geemap import ml
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/land_cover_classification_kaza

In [ ]:
user_id = "ee-alexvmt"
asset_name = "mufunta_random_forest"

In [ ]:
ee.Authenticate()
ee.Initialize(project=user_id)

## Convert sklearn classifier object to a list of strings

In [ ]:
model = ml.load_model("models/random_forest_model.joblib")
feature_names = model.feature_names_in_.tolist()

In [ ]:
# convert the estimator into a list of strings
# this function also works with the ensemble.ExtraTrees estimator
start_time = time.perf_counter()
trees = ml.rf_to_strings(model, feature_names)
end_time = time.perf_counter()
run_time = round((end_time - start_time) / 60, 2)
print("Run time: {} minutes.".format(run_time))

## Convert sklearn classifier to GEE classifier

At this point you can take the list of strings and save them locally to avoid training again. However, we want to use the model with EE so we need to create an ee.Classifier and persist the data on ee for best results.

In [ ]:
# create a ee classifier to use with ee objects from the trees
ee_classifier = ml.strings_to_classifier(trees)

## Save trees to the cloud

Now we have the strings in a format that ee can use, we want to save it for later use. There is a function to export a list of tree strings to a feature collection.

In [ ]:
# specify asset id where to save trees
asset_id = "projects/" + user_id + "/assets/" + asset_name + "_trees"
asset_id

In [ ]:
# kick off an export process so it will be saved to the ee asset
ml.export_trees_to_fc(trees, asset_id)

# this will kick off an export task, so wait a few minutes before moving on
# check progress here: https://code.earthengine.google.com/tasks

In [ ]:
# save ee classifier to be used in ee directly
classifier_asset_id = "projects/" + user_id + "/assets/" + asset_name + "_classifier"
task = ee.batch.Export.classifier.toAsset(ee_classifier, "saved classifier", classifier_asset_id)
task.start()